In [3]:
# ============================================================
# NB05_Paper_Portfolio (₹10 LAKHS RETAIL PORTFOLIO)
# Simulates buying Lynch candidates and tracks holdings over time.
#
# Reads:  gold_slow_grower_candidates (latest screen_date)
#         gold_stalwart_candidates    (latest screen_date)
#         bronze_yahoo_prices         (latest close)
# Writes: gold_lynch_portfolio   (current holdings)
#         gold_lynch_transactions (all buys, sells, dividends)
# ============================================================

import uuid
import numpy as np
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    DoubleType, IntegerType, LongType, TimestampType
)

RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
RUN_ID = str(uuid.uuid4())
SCREEN_DATE = datetime.utcnow().date()

# ------------------------------------------------------------
# 1. Configuration (EXACTLY ₹10 LAKHS)
# ------------------------------------------------------------
INITIAL_CAPITAL = 1000000.0   # Exactly ₹10,00,000.00 (10 Lakhs)
MAX_PER_STOCK_PCT = 0.05      # 5% max per stock (₹50,000 per stock)

print("=" * 60)
print(f"PORTFOLIO CONFIGURATION")
print("=" * 60)
print(f"Starting Capital  : ₹{INITIAL_CAPITAL:,.2f} (10 Lakhs)")
print(f"Max Allocation/Stk: ₹{INITIAL_CAPITAL * MAX_PER_STOCK_PCT:,.2f} (5% max)")
print("=" * 60)

# ------------------------------------------------------------
# 2. Load latest candidates from both screens
# ------------------------------------------------------------
def load_latest_candidates(table_name):
    try:
        table_df = spark.table(table_name)
        if table_df.count() > 0:
            df = spark.sql(f"""
                SELECT symbol, company_name, candidate_rank, screen_date, candidate_type
                FROM {table_name}
                WHERE screen_date = (SELECT MAX(screen_date) FROM {table_name})
            """).toPandas()
            df = df.loc[:, ~df.columns.duplicated()]
            return df
        return pd.DataFrame(columns=["symbol", "company_name", "candidate_rank", "screen_date", "candidate_type"])
    except Exception as e:
        print(f"Note on {table_name}: {e}")
        return pd.DataFrame(columns=["symbol", "company_name", "candidate_rank", "screen_date", "candidate_type"])

slow_candidates = load_latest_candidates("gold_slow_grower_candidates")
stal_candidates = load_latest_candidates("gold_stalwart_candidates")

print(f"\nSlow Grower candidates: {len(slow_candidates)}")
print(f"Stalwart candidates   : {len(stal_candidates)}")

candidate_frames = [df for df in [slow_candidates, stal_candidates] if not df.empty]

if candidate_frames:
    latest_screen = pd.concat(candidate_frames, ignore_index=True)
    # Deduplicate by symbol in case a stock qualified in both categories
    latest_screen["symbol"] = latest_screen["symbol"].astype(str).str.strip().str.upper()
    latest_screen = latest_screen.drop_duplicates(subset=["symbol"], keep="first").reset_index(drop=True)
else:
    latest_screen = pd.DataFrame()

print(f"Combined unique candidates to allocate: {len(latest_screen)}")

# ------------------------------------------------------------
# 3. Load existing portfolio (Reset if old balance was multi-Crore)
# ------------------------------------------------------------
cash_balance = INITIAL_CAPITAL
existing_portfolio = pd.DataFrame()

try:
    existing_spark = spark.table("gold_lynch_portfolio")
    if existing_spark.count() > 0:
        existing_portfolio = existing_spark.toPandas()
        old_cash = float(existing_portfolio["cash_balance"].iloc[0]) if "cash_balance" in existing_portfolio.columns and not existing_portfolio.empty else 0.0
        
        # If old cash was the previous huge multi-Crore test balance, reset cleanly to 10 Lakhs
        if old_cash > 2000000.0:  # If greater than 20 Lakhs
            print("\n⚠️ Detected previous multi-Crore test run. Resetting portfolio cleanly to ₹10 Lakhs.")
            cash_balance = INITIAL_CAPITAL
            existing_portfolio = pd.DataFrame()
        else:
            cash_balance = old_cash
            print(f"\nExisting portfolio active. Available Cash: ₹{cash_balance:,.2f}")
except Exception:
    print("\nStarting fresh portfolio.")

# ------------------------------------------------------------
# 4. Load current prices for candidates
# ------------------------------------------------------------
prices = (
    spark.sql("""
        SELECT symbol, trade_date, close
        FROM bronze_yahoo_prices
        WHERE trade_date = (SELECT MAX(trade_date) FROM bronze_yahoo_prices)
          AND close IS NOT NULL AND close > 0
    """)
    .toPandas()
)

price_map = dict(zip(prices["symbol"].astype(str), prices["close"].astype(float)))
print(f"Price lookups loaded for {len(price_map)} symbols")

# ------------------------------------------------------------
# 5. Simulate buys
# ------------------------------------------------------------
transactions = []
portfolio_holdings = []

available_cash = cash_balance
num_candidates = len(latest_screen)

target_position = min(
    INITIAL_CAPITAL * MAX_PER_STOCK_PCT,
    available_cash / max(num_candidates, 1)
)

print(f"Target position size: ₹{target_position:,.2f} per stock\n")

if not latest_screen.empty:
    for _, row in latest_screen.iterrows():
        symbol = str(row["symbol"])
        company = str(row.get("company_name", symbol))
        cand_type = str(row.get("candidate_type", "STALWART"))
        current_price = price_map.get(symbol)
        
        if current_price is None or current_price <= 0:
            print(f"  ⏭️ {symbol}: No price available, skipping")
            continue
        
        # Check if already holding
        if not existing_portfolio.empty and "symbol" in existing_portfolio.columns:
            already_held = existing_portfolio[existing_portfolio["symbol"] == symbol]
            if not already_held.empty and int(already_held["quantity"].iloc[0]) > 0:
                print(f"  ⏭️ {symbol}: Already held in portfolio, skipping")
                continue
        
        # Calculate quantity based on retail position size (₹50,000)
        quantity = int(target_position // current_price)
        if quantity <= 0:
            print(f"  ⏭️ {symbol}: Share price (₹{current_price:,.2f}) exceeds allocation, skipping")
            continue
        
        cost = quantity * current_price
        if cost > available_cash:
            print(f"  ⚠️ {symbol}: Insufficient cash remaining")
            continue
        
        # Record transaction
        transactions.append({
            "transaction_id": str(uuid.uuid4()),
            "transaction_date": str(SCREEN_DATE),
            "transaction_type": "BUY",
            "symbol": symbol,
            "quantity": quantity,
            "price": float(current_price),
            "amount": float(cost),
            "notes": f"Type: {cand_type}, Rank: {row.get('candidate_rank', '-')}",
            "screen_date": str(SCREEN_DATE),
            "run_id": RUN_ID
        })
        
        # Update portfolio
        portfolio_holdings.append({
            "symbol": symbol,
            "company_name": company,
            "candidate_type": cand_type,
            "quantity": quantity,
            "avg_cost": float(current_price),
            "current_price": float(current_price),
            "market_value": float(cost),
            "unrealized_pnl": 0.0,
            "unrealized_pnl_pct": 0.0,
            "weight_pct": 0.0,
            "screen_date": str(SCREEN_DATE),
            "run_id": RUN_ID
        })
        
        available_cash -= cost
        print(f"  🟢 BOUGHT {quantity} shares of {symbol} ({cand_type}) @ ₹{current_price:,.2f} = ₹{cost:,.2f}")

# ------------------------------------------------------------
# 6. Calculate portfolio metrics
# ------------------------------------------------------------
total_market_value = 0.0
total_value = available_cash

if portfolio_holdings:
    portfolio_df = pd.DataFrame(portfolio_holdings)
    total_market_value = portfolio_df["market_value"].sum()
    total_value = total_market_value + available_cash
    
    portfolio_df["weight_pct"] = (portfolio_df["market_value"] / total_market_value * 100).round(2)
    portfolio_df["cash_balance"] = round(available_cash, 2)
    portfolio_df["total_value"] = round(total_value, 2)
else:
    portfolio_df = pd.DataFrame(columns=[
        "symbol", "company_name", "candidate_type", "quantity", "avg_cost",
        "current_price", "market_value", "unrealized_pnl",
        "unrealized_pnl_pct", "weight_pct", "cash_balance", "total_value",
        "screen_date", "run_id"
    ])

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------
portfolio_cols = [
    "symbol", "company_name", "candidate_type", "quantity", "avg_cost",
    "current_price", "market_value", "unrealized_pnl",
    "unrealized_pnl_pct", "weight_pct", "cash_balance", "total_value",
    "screen_date", "run_id"
]

if not portfolio_df.empty:
    (
        spark.createDataFrame(portfolio_df[portfolio_cols])
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("gold_lynch_portfolio")
    )
    print(f"\n✅ Saved gold_lynch_portfolio: {len(portfolio_df)} holdings")
else:
    print("\nNo holdings. Initialized empty gold_lynch_portfolio table.")

if transactions:
    tx_df = pd.DataFrame(transactions)
    (
        spark.createDataFrame(tx_df)
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("gold_lynch_transactions")
    )
    print(f"✅ Saved {len(transactions)} transactions to gold_lynch_transactions")
else:
    print("No transactions executed this run.")

# ------------------------------------------------------------
# 8. Summary (₹ Lakhs Formatting)
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("PORTFOLIO SUMMARY (RETAIL SCALE)")
print("=" * 60)
print(f"Total Account Value : ₹{total_value:,.2f}")
print(f"Invested Capital    : ₹{total_market_value:,.2f}")
print(f"Cash Remaining      : ₹{available_cash:,.2f}")
print(f"Active Positions    : {len(portfolio_df)}")
print("=" * 60)

if not portfolio_df.empty:
    display(portfolio_df[["symbol", "company_name", "candidate_type", "quantity", "avg_cost", "market_value", "weight_pct"]])

StatementMeta(, 756cbc7d-b37b-4da6-98cf-f95efe3311b2, 5, Finished, Available, Finished, False)

PORTFOLIO CONFIGURATION
Starting Capital  : ₹1,000,000.00 (10 Lakhs)
Max Allocation/Stk: ₹50,000.00 (5% max)

Slow Grower candidates: 4
Stalwart candidates   : 11
Combined unique candidates to allocate: 15

Existing portfolio active. Available Cash: ₹277,450.61
Price lookups loaded for 352 symbols
Target position size: ₹18,496.71 per stock

  ⏭️ ONGC: Already held in portfolio, skipping
  ⏭️ PETRONET: Already held in portfolio, skipping
  ⏭️ HCLTECH: Already held in portfolio, skipping
  ⏭️ WIPRO: Already held in portfolio, skipping
  ⏭️ OFSS: Already held in portfolio, skipping
  ⏭️ SUNPHARMA: Already held in portfolio, skipping
  ⏭️ ECLERX: Already held in portfolio, skipping
  ⏭️ HINDUNILVR: Already held in portfolio, skipping
  ⏭️ COLPAL: Already held in portfolio, skipping
  ⏭️ LTFOODS: Already held in portfolio, skipping
  ⏭️ BSOFT: Already held in portfolio, skipping
  ⏭️ BERGEPAINT: Already held in portfolio, skipping
  ⏭️ ELECON: Already held in portfolio, skipping
  ⏭️ USHAMA